# NewsBot Intelligence System 2.0 — Final executable notebook
José Miguel Tuozzo Izaguirre · ITAI 2373 · Houston Community College

This notebook runs the actual final library. The 30 records are fictional; historical BBC results are reported separately. Optional neural evidence is produced by the separate multilingual job.

In [1]:
from pathlib import Path
import sys, json, os
root = Path.cwd()
if not (root / 'newsbot2').is_dir():
    root = next(p for p in (root/'final-newsbot-2.0',root.parent) if (p/'newsbot2').is_dir())
os.chdir(root)
sys.path.insert(0,str(root))
from newsbot2.io import import_articles
from newsbot2.engine import NewsBotEngine
from newsbot2.evaluation import evaluate_classification_cv
from IPython.display import display
import pandas as pd
articles = import_articles(root/'data/demo_news.jsonl')
engine = NewsBotEngine(articles)
assert len(articles)==30 and all(a.synthetic for a in articles)
display(engine.overview())

{'total_articles': 30,
 'label_distribution': {'tech': 6,
  'business': 6,
  'politics': 6,
  'sport': 6,
  'entertainment': 6},
 'language_hints': {'en': 25, 'es': 2, 'fr': 2, 'de': 1},
 'synthetic_articles': 30,
 'retrieval_method': 'tfidf-pivot',
 'classifier_origin': 'not-trained',
 'data_warning': 'Synthetic examples are demos, not real-world validation.'}

## 1. Advanced content analysis
Inspect topics, category fit, sentiment, entity candidates and provenance. Local fit is not independent validation.

In [2]:
display(engine.train_local_classifier())
analysis=engine.analyze('demo-tech-1')
display({k:analysis[k] for k in ('classification','sentiment','entities','provenance')})
topics=engine.topics(5)
display(pd.DataFrame(topics['topics']))

{'status': 'trained',
 'training_examples': 30,
 'training_label_counts': {'tech': 6,
  'business': 6,
  'politics': 6,
  'sport': 6,
  'entertainment': 6},
 'warning': 'Training fit is NOT a holdout metric'}

{'classification': {'status': 'ok',
  'label': 'tech',
  'ranked_scores': [{'category': np.str_('tech'), 'score': 0.3777},
   {'category': np.str_('sport'), 'score': 0.1584},
   {'category': np.str_('business'), 'score': 0.1552},
   {'category': np.str_('politics'), 'score': 0.1544},
   {'category': np.str_('entertainment'), 'score': 0.1542}],
  'score_type': 'model-estimated-probabilities; uncalibrated',
  'top_margin_gap': 0.2193,
  'training_origin': 'current-local-corpus (not externally evaluated)',
  'warning': 'Category suggestion; verify ambiguous and shifted-domain articles'},
 'sentiment': {'label': 'neutral',
  'score': 0.0,
  'method': 'weak-lexicon-fallback',
  'warning': 'Exploratory language-dependent lexical indicator, NOT validated sentiment'},
 'entities': {'mentions': [],
  'method': 'regex-candidates',
  'warning': 'Candidate phrases only; not named-entity recognition'},
 'provenance': {'article_id': 'demo-tech-1',
  'sha256': 'c4e7d2bf53009bc237f26731d2842a082e621cf

,topic_id,terms,article_count_top_topic
0,0,"[league, youth, schedule, tournament, football...",2
1,1,"[training, local, digital, event, swimming, cl...",9
2,2,"[software, team, tested, researchers, power, p...",7
3,3,"[le, une, les, cinéma, programme, des, gouvern...",4
4,4,"[school, public, council, student, proposal, b...",8


## 2. Language understanding and generation
The summarizer selects source sentences; headline suggestions use templates. Compare sparse and latent semantic retrieval.

In [3]:
display(analysis['summary'],analysis['enhancement'])
lsa=NewsBotEngine(articles,retrieval_mode='lsa')
for e in (engine,lsa):
    display(pd.DataFrame(e.search('robot cameras sensors navigation',k=3))[['article_id','score','retrieval_method']])

{'text': 'A university engineering team tested an experimental computer chip designed to reduce the energy required for artificial intelligence tasks. The study measured processing time and power consumption under laboratory conditions. The researchers plan another trial before any public release.',
 'method': 'extractive',
 'selected_sentence_indices': [0, 1, 2]}

{'headlines': ['Engineers test low-power AI chips',
  'Briefing: Engineers test low-power AI chips',
  'Explainer | Engineers test low-power AI chips'],
 'short_blurb': 'A university engineering team tested an experimental computer chip designed to reduce the energy required for artificial intelligence tasks. The study measured processing time and power consumption under laboratory conditions. The researchers plan another trial before any public release.',
 'guardrail': 'Templates preserve user-supplied title; editor must verify accuracy'}

,article_id,score,retrieval_method
0,demo-tech-2,0.2927,tfidf-pivot
1,demo-tech-6,0.0514,tfidf-pivot


,article_id,score,retrieval_method
0,demo-tech-2,0.9871,lsa-svd-pivot
1,demo-tech-6,0.1733,lsa-svd-pivot
2,demo-entertainment-2,0.0142,lsa-svd-pivot


## 3. Multilingual intelligence
Default lexical pivot is limited; the separate neural test installs real weights and translation packs.

In [4]:
from newsbot2.language import detect_language
for query in ['investigación de inteligencia artificial','technologie robotique','forschung daten']:
    display(query,detect_language(query),pd.DataFrame(engine.search(query,k=3)))
display(engine.search('robótica navegación universidad',k=3,language='es'))
display(engine.translate('La universidad investiga inteligencia artificial.','es','en'))

'investigación de inteligencia artificial'

{'language': 'und',
 'confidence': 'low',
 'method': 'stopword-heuristic',
 'scores': {'en': 0, 'es': 1, 'fr': 1, 'de': 0}}

,article_id,title,score,excerpt,source,synthetic,retrieval_method
0,demo-tech-6,Universidad investiga robótica para navegación,0.1807,Un equipo de la universidad desarrolló un robo...,Synthetic classroom demonstration — fictional,True,tfidf-pivot
1,demo-tech-1,Engineers test low-power AI chips,0.1136,A university engineering team tested an experi...,Synthetic classroom demonstration — fictional,True,tfidf-pivot
2,demo-business-2,Small manufacturers review factory energy costs,0.0495,Manufacturers met with an economic research as...,Synthetic classroom demonstration — fictional,True,tfidf-pivot


'technologie robotique'

{'language': 'und',
 'confidence': 'low',
 'method': 'stopword-heuristic',
 'scores': {'en': 0, 'es': 0, 'fr': 0, 'de': 0}}

,article_id,title,score,excerpt,source,synthetic,retrieval_method
0,demo-tech-2,Robotics lab demonstrates safer indoor navigation,0.1356,A robotics laboratory presented an indoor robo...,Synthetic classroom demonstration — fictional,True,tfidf-pivot
1,demo-politics-5,Council considers school budget hearings,0.0885,A fictional municipal council announced public...,Synthetic classroom demonstration — fictional,True,tfidf-pivot
2,demo-business-5,Local firms invest in employee training,0.0841,Several fictional technology suppliers announc...,Synthetic classroom demonstration — fictional,True,tfidf-pivot


'forschung daten'

{'language': 'und',
 'confidence': 'low',
 'method': 'stopword-heuristic',
 'scores': {'en': 0, 'es': 0, 'fr': 0, 'de': 0}}

,article_id,title,score,excerpt,source,synthetic,retrieval_method
0,demo-tech-3,Researchers explore privacy-preserving data tools,0.2371,Computer scientists published a simulated stud...,Synthetic classroom demonstration — fictional,True,tfidf-pivot
1,demo-business-2,Small manufacturers review factory energy costs,0.0692,Manufacturers met with an economic research as...,Synthetic classroom demonstration — fictional,True,tfidf-pivot
2,demo-tech-5,Startup team prototypes satellite mapping soft...,0.0649,A startup research team tested satellite image...,Synthetic classroom demonstration — fictional,True,tfidf-pivot


[{'article_id': 'demo-tech-6',
  'title': 'Universidad investiga robótica para navegación',
  'score': 0.2998,
  'excerpt': 'Un equipo de la universidad desarrolló un robot experimental para la navegación autónoma. Los investigadores midieron sensores de distancia y nuevas técnicas de inteligencia artificial.',
  'source': 'Synthetic classroom demonstration — fictional',
  'synthetic': True,
  'retrieval_method': 'tfidf-pivot'}]

{'status': 'unavailable',
 'translated_text': None,
 'provider': 'argos-local',
 'reason': 'Argos or language pack unavailable: ModuleNotFoundError'}

## 4. Conversation and abstention
Only source excerpts are returned. Relevance is not truth.

In [5]:
answer=engine.ask('What did the AI chip researchers measure?')
display(answer)
assert answer['citations']
display(engine.ask('Tell me more'))
unknown=engine.ask('zzqxv nonexistent topic 0123456789')
display(unknown)
assert unknown['status']=='insufficient_evidence'

{'status': 'grounded_extract',
 'answer': '[demo-tech-1] A university engineering team tested an experimental computer chip designed to reduce the energy required for artificial intelligence tasks. The study measured processing time and power consumption under laboratory conditions.',
 'citations': [{'article_id': 'demo-tech-1',
   'title': 'Engineers test low-power AI chips',
   'source': 'Synthetic classroom demonstration — fictional',
   'retrieval_score': 0.185,
   'synthetic': True}],
 'retrieval_method': 'tfidf-pivot',
 'confidence_policy': 'retrieval-score-only (not calibrated answer probability)',
 'warning': 'These excerpts come from source texts; they are not independently fact-checked.',
 'resolved_query': 'What did the AI chip researchers measure?'}

{'status': 'grounded_extract',
 'answer': '[demo-tech-1] A university engineering team tested an experimental computer chip designed to reduce the energy required for artificial intelligence tasks. The study measured processing time and power consumption under laboratory conditions.',
 'citations': [{'article_id': 'demo-tech-1',
   'title': 'Engineers test low-power AI chips',
   'source': 'Synthetic classroom demonstration — fictional',
   'retrieval_score': 0.4471,
   'synthetic': True}],
 'retrieval_method': 'tfidf-pivot',
 'confidence_policy': 'retrieval-score-only (not calibrated answer probability)',
 'warning': 'These excerpts come from source texts; they are not independently fact-checked.',
 'resolved_query': 'Tell me more Engineers test low-power AI chips'}

{'status': 'insufficient_evidence',
 'answer': 'I could not find adequate supporting material in the loaded articles. Please name a topic, entity or article title, or upload relevant sources.',
 'citations': [],
 'retrieval_method': 'tfidf-pivot',
 'confidence_policy': 'abstain-on-low-retrieval',
 'resolved_query': 'zzqxv nonexistent topic 0123456789'}

## 5. Experimental evaluation
Stratified out-of-fold prediction fits features inside training folds. Controlled synthetic benchmark only.

In [6]:
cv=evaluate_classification_cv(articles,folds=3)
display(pd.DataFrame(cv['per_class']).T)
display(pd.DataFrame(cv['confusion_matrix'],index=cv['classes'],columns=cv['classes']))
import subprocess
subprocess.run([sys.executable,'run_research.py'],check=True)
research=json.loads((root/'artifacts/research_results.json').read_text())
display({k:research['retrieval_probes'][k] for k in ('n','k','pivot_mrr','naive_mrr','pivot_recall_at_k','naive_recall_at_k')})
display(research['paired_multilingual_extension'])

,precision,recall,f1-score,support
business,0.500000,0.166667,0.250000,6.000000
entertainment,0.400000,0.333333,0.363636,6.000000
politics,0.444444,0.666667,0.533333,6.000000
sport,0.714286,0.833333,0.769231,6.000000
tech,0.714286,0.833333,0.769231,6.000000
accuracy,0.566667,0.566667,0.566667,0.566667
macro avg,0.554603,0.566667,0.537086,30.000000
weighted avg,0.554603,0.566667,0.537086,30.000000


,business,entertainment,politics,sport,tech
business,1,0,2,1,2
entertainment,0,2,3,1,0
politics,1,1,4,0,0
sport,0,1,0,5,0
tech,0,1,0,0,5


/opt/hostedtoolcache/Python/3.11.18/x64/lib/python3.11/site-packages/sklearn/metrics/_classification.py:614: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/opt/hostedtoolcache/Python/3.11.18/x64/lib/python3.11/site-packages/sklearn/metrics/_classification.py:614: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(


/opt/hostedtoolcache/Python/3.11.18/x64/lib/python3.11/site-packages/sklearn/metrics/_classification.py:614: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(


Saved artifacts/research_results.json
Retrieval MRR@3 pivot / no pivot: 1.0 0.625
Synthetic classification CV macro-F1: 0.5371
{
  "baseline": {
    "accuracy": 0.5666666666666667,
    "macro_f1": 0.5370862470862471
  },
  "pivot": {
    "accuracy": 0.6,
    "macro_f1": 0.5719780219780219
  },
  "pivot_char": {
    "accuracy": 0.6666666666666666,
    "macro_f1": 0.6142857142857143
  }
}


{'n': 24,
 'k': 3,
 'pivot_mrr': 1.0,
 'naive_mrr': 0.625,
 'pivot_recall_at_k': 1.0,
 'naive_recall_at_k': 0.625}

{'synthetic_only': True,
 'n': 30,
 'labels': ['business', 'entertainment', 'politics', 'sport', 'tech'],
 'language_counts': {'en': 25, 'es': 2, 'fr': 2, 'de': 1},
 'seed': 42,
 'protocol': 'Same stratified folds for all models; fit feature extraction inside each fold; language holdout is distribution-shift diagnostic.',
 'paired_oof': {'baseline': {'accuracy': 0.5666666666666667,
   'macro_f1': 0.5370862470862471,
   'confusion_matrix': [[1, 0, 2, 1, 2],
    [0, 2, 3, 1, 0],
    [1, 1, 4, 0, 0],
    [0, 1, 0, 5, 0],
    [0, 1, 0, 0, 5]],
   'elapsed_seconds': 0.0382,
   'predictions': [{'id': 'demo-tech-1', 'gold': 'tech', 'predicted': 'tech'},
    {'id': 'demo-tech-2', 'gold': 'tech', 'predicted': 'tech'},
    {'id': 'demo-tech-3', 'gold': 'tech', 'predicted': 'tech'},
    {'id': 'demo-tech-4', 'gold': 'tech', 'predicted': 'tech'},
    {'id': 'demo-tech-5', 'gold': 'tech', 'predicted': 'tech'},
    {'id': 'demo-business-1', 'gold': 'business', 'predicted': 'business'},
    {'id': 'd

## 6. Engineering evidence
Automated checks, browser captures and neural evidence are maintained independently. Scientific limitations: selected fictional probes, no independent real multilingual benchmark, uncalibrated scores, no factual verification.

In [7]:
subprocess.run([sys.executable,'-m','pytest','tests','-q'],check=True)
print('Notebook completed from a fresh kernel.')

........

...................                                              [100%]


=============================== warnings summary ===============================
../../../../../../opt/hostedtoolcache/Python/3.11.18/x64/lib/python3.11/site-packages/fastapi/testclient.py:1
  /opt/hostedtoolcache/Python/3.11.18/x64/lib/python3.11/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
    from starlette.testclient import TestClient as TestClient  # noqa

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
27 passed, 1 warning in 1.84s


Notebook completed from a fresh kernel.
